# Effect of Force-Level Conditioning

This notebook shows the change from the object and object-region final evaluation models to the corresponding force-level-conditioned models. Accuracy changes are percentage points, while weighted F1 changes use the original 0–1 scale.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Paths and experiment numbers
RESULTS_DIR = Path("../results/final_evaluation")
OUTPUT_DIR = Path("../exports")
MAIN_EXPERIMENT_NUMBER = 1
CONDITIONED_EXPERIMENT_NUMBER = 1

CONDITIONING_METRIC_COLUMNS = [
    "test_accuracy",
    "test_macro_f1_avg",
    "test_unseen_matched_accuracy",
    "test_unseen_matched_macro_f1_avg",
]


def load_conditioning_results(label):
    """
    Load the Main and conditioned results for a given label.

    Args:
        label (str): The label for which to load results.

    Returns:
        tuple[pd.Series, pd.Series]: The Main and conditioned result rows.
    """

    main_path = RESULTS_DIR / f"{label}_classification" / "experiments.parquet"
    main = (
        pd.read_parquet(main_path)
        .query(f"experiment_number == {MAIN_EXPERIMENT_NUMBER}")
        .iloc[0]
    )
    conditioned_path = (
        RESULTS_DIR / f"{label}_conditioned_classification" / "experiments.parquet"
    )
    conditioned = (
        pd.read_parquet(conditioned_path)
        .query(f"experiment_number == {CONDITIONED_EXPERIMENT_NUMBER}")
        .iloc[0]
    )
    return main, conditioned


def calculate_conditioning_deltas(main, conditioned):
    """
    Calculate signed conditioned-minus-Main metric changes.

    Args:
        main (pd.Series): The image-only final evaluation result row.
        conditioned (pd.Series): The conditioned result row.

    Returns:
        dict: The signed change for each comparison metric.
    """

    deltas = conditioned[CONDITIONING_METRIC_COLUMNS] - main[CONDITIONING_METRIC_COLUMNS]
    return {f"{metric}_delta": value for metric, value in deltas.items()}


def export_conditioning_comparison():
    """
    Export conditioned-minus-Main metric changes for both labels.

    Returns:
        pd.DataFrame: A DataFrame containing the conditioned-minus-Main metric changes.
    """

    rows = []
    for label in ["object", "object_region"]:
        main, conditioned = load_conditioning_results(label)
        deltas = calculate_conditioning_deltas(main, conditioned)
        row = {"label": label}
        row.update(
            {
                metric: f"{value:+.4f}" if "f1" in metric else f"{value:+.2f}"
                for metric, value in deltas.items()
            }
        )
        rows.append(row)

    # Create a DataFrame from the rows and export it to CSV
    comparison = pd.DataFrame(rows)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(OUTPUT_DIR / "conditioning_comparison.csv", index=False)
    return comparison


In [ ]:
def plot_conditioning_comparison(comparison):
    """
    Plot and save the conditioned-minus-main metric changes.

    Args:
        comparison (pd.DataFrame): A DataFrame containing the comparison data.
    """

    # Set the index to the label column and convert all values to float for plotting
    plot_data = comparison.set_index("label").astype(float)

    # Express macro F1 changes in percentage points, matching accuracy
    f1_columns = [column for column in plot_data if "f1" in column]
    plot_data[f1_columns] *= 100

    # Use readable legend labels instead of stored identifiers
    plot_data = plot_data.rename(
        index={
            "object": "Object",
            "object_region": "Object Region",
        }
    )

    plot_data.columns = [
        "Internal Test Accuracy",
        "Internal Test Macro F1",
        "Unseen Matched Accuracy",
        "Unseen Matched Macro F1",
    ]

    # Create a bar plot with a dark grid style
    with sns.axes_style("darkgrid"):
        fig, ax = plt.subplots(figsize=(14, 5))
        plot_data.T.plot.bar(
            ax=ax,
            color=sns.color_palette("deep", n_colors=len(plot_data)),
        )

    ax.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=0.8,
    )
    ax.set_title(
        "Effect of Force-Level Conditioning",
        fontsize=14,
        fontweight="normal",
    )
    ax.set_ylabel("Conditioned − Unconditioned (% Points)")
    ax.set_xlabel("")
    ax.legend(title="Label")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    fig.tight_layout()

    # Save the plot
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        OUTPUT_DIR / "conditioning_comparison.png",
        dpi=300,
        bbox_inches="tight",
    )

In [ ]:
conditioning_comparison = export_conditioning_comparison()
conditioning_comparison

In [ ]:
plot_conditioning_comparison(conditioning_comparison)